# 02 - Task 1 models (handcrafted vs frozen CNN)

This notebook trains and compares four Task 1 models using:

- an 80/20 stratified holdout split
- 5-fold stratified CV on the 80% training partition

Models:

1. Logistic Regression on handcrafted features
2. RBF SVM on handcrafted features
3. Random Forest on handcrafted features
4. Logistic Regression on ResNet18 embeddings

Outputs written to `outputs/`:

- experiment rows in `outputs/metrics/log.csv`
- confusion matrices in `outputs/figures/task1/`
- predictions in `outputs/predictions/`

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from utils import (
    evaluate,
    kfold,
    load_embedding_cache,
    load_task,
    log_experiment,
    make_submission,
    plot_confusion,
    seed_everything,
    stratified_split,
)

SEED = 42
seed_everything(SEED)

OUT_PRED = REPO_ROOT / "outputs" / "predictions"
OUT_FIG = REPO_ROOT / "outputs" / "figures" / "task1"
OUT_PRED.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 80)

In [2]:
bundle = load_task(1)

X_hand_train = bundle.X_handcrafted_train
X_hand_test = bundle.X_handcrafted_test
y = bundle.y_train
train_ids = bundle.train_ids
test_ids = bundle.test_ids
class_mapping = bundle.class_mapping.sort_values("class_id").reset_index(drop=True)
class_ids = class_mapping["class_id"].tolist()
class_names = class_mapping["class_name"].tolist()
id_to_name = dict(zip(class_mapping["class_id"], class_mapping["class_name"]))

X_resnet_train, resnet_train_ids = load_embedding_cache(1, "train", "resnet18")
X_resnet_test, resnet_test_ids = load_embedding_cache(1, "test", "resnet18")

assert list(resnet_train_ids) == list(train_ids)
assert list(resnet_test_ids) == list(test_ids)

Xh_tr, Xh_val, y_tr, y_val, ids_tr, ids_val = stratified_split(
    X_hand_train,
    y,
    ids=train_ids,
    test_size=0.2,
    seed=SEED,
)

Xr_tr, Xr_val, _, _, _, _ = stratified_split(
    X_resnet_train,
    y,
    ids=train_ids,
    test_size=0.2,
    seed=SEED,
)

print(f"Task 1 train size: {len(y)}")
print(f"Holdout split: train={len(y_tr)} | val={len(y_val)}")
print(f"Handcrafted dim: {X_hand_train.shape[1]} | ResNet18 dim: {X_resnet_train.shape[1]}")

Task 1 train size: 3750
Holdout split: train=3000 | val=750
Handcrafted dim: 219 | ResNet18 dim: 512


In [3]:
model_specs = {
    "lr_handcrafted": {
        "feature_set": "handcrafted_219d",
        "X_train_holdout": Xh_tr,
        "X_val_holdout": Xh_val,
        "X_train_full": X_hand_train,
        "X_test": X_hand_test,
        "estimator": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=2000,
                        C=1.0,
                        solver="lbfgs",
                        multi_class="auto",
                        random_state=SEED,
                    ),
                ),
            ]
        ),
    },
    "svm_rbf_handcrafted": {
        "feature_set": "handcrafted_219d",
        "X_train_holdout": Xh_tr,
        "X_val_holdout": Xh_val,
        "X_train_full": X_hand_train,
        "X_test": X_hand_test,
        "estimator": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "clf",
                    SVC(
                        C=10.0,
                        gamma="scale",
                        kernel="rbf",
                        random_state=SEED,
                    ),
                ),
            ]
        ),
    },
    "rf_handcrafted": {
        "feature_set": "handcrafted_219d",
        "X_train_holdout": Xh_tr,
        "X_val_holdout": Xh_val,
        "X_train_full": X_hand_train,
        "X_test": X_hand_test,
        "estimator": RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=1,
            n_jobs=-1,
            random_state=SEED,
        ),
    },
    "lr_resnet18": {
        "feature_set": "resnet18_512d",
        "X_train_holdout": Xr_tr,
        "X_val_holdout": Xr_val,
        "X_train_full": X_resnet_train,
        "X_test": X_resnet_test,
        "estimator": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=3000,
                        C=1.0,
                        solver="lbfgs",
                        multi_class="auto",
                        random_state=SEED,
                    ),
                ),
            ]
        ),
    },
}

results = []
trained_models = {}

for model_name, spec in model_specs.items():
    print(f"\n=== {model_name} ===")
    X_tr_local = spec["X_train_holdout"]
    X_val_local = spec["X_val_holdout"]
    est = spec["estimator"]

    cv = cross_validate(
        clone(est),
        X_tr_local,
        y_tr,
        cv=kfold(5, seed=SEED),
        scoring={"acc": "accuracy", "macro_f1": "f1_macro"},
        return_train_score=False,
        n_jobs=-1,
    )

    t0 = time.perf_counter()
    fitted = clone(est)
    fitted.fit(X_tr_local, y_tr)
    train_time_s = time.perf_counter() - t0

    y_val_pred = fitted.predict(X_val_local)
    metrics = evaluate(y_val, y_val_pred, labels=class_ids)

    cm_path = OUT_FIG / f"{model_name}_holdout_confusion.png"
    plot_confusion(
        metrics["confusion_matrix"],
        labels=class_names,
        out_path=cm_path,
        title=f"Task 1 holdout confusion - {model_name}",
        normalize=True,
    )

    log_row = {
        "task": 1,
        "model": model_name,
        "feature_set": spec["feature_set"],
        "hyperparams": json.dumps(est.get_params(), default=str),
        "cv_mean_acc": float(np.mean(cv["test_acc"])),
        "cv_std_acc": float(np.std(cv["test_acc"])),
        "cv_mean_macro_f1": float(np.mean(cv["test_macro_f1"])),
        "cv_std_macro_f1": float(np.std(cv["test_macro_f1"])),
        "val_acc": float(metrics["accuracy"]),
        "val_macro_f1": float(metrics["macro_f1"]),
        "train_time_s": float(train_time_s),
        "notes": "task1_holdout80_cv5",
    }
    log_experiment(log_row)

    results.append(
        {
            "model": model_name,
            "feature_set": spec["feature_set"],
            "cv_acc_mean": log_row["cv_mean_acc"],
            "cv_acc_std": log_row["cv_std_acc"],
            "cv_macro_f1_mean": log_row["cv_mean_macro_f1"],
            "cv_macro_f1_std": log_row["cv_std_macro_f1"],
            "holdout_acc": log_row["val_acc"],
            "holdout_macro_f1": log_row["val_macro_f1"],
            "fit_time_s": log_row["train_time_s"],
            "confusion_png": str(cm_path.relative_to(REPO_ROOT)),
        }
    )

    trained_models[model_name] = fitted
    print(f"CV acc={log_row['cv_mean_acc']:.4f} +/- {log_row['cv_std_acc']:.4f}")
    print(f"CV f1 ={log_row['cv_mean_macro_f1']:.4f} +/- {log_row['cv_std_macro_f1']:.4f}")
    print(f"Holdout acc={log_row['val_acc']:.4f} | macro-F1={log_row['val_macro_f1']:.4f}")
    print(f"Saved confusion matrix: {cm_path}")


=== lr_handcrafted ===


/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/li

CV acc=0.5010 +/- 0.0164
CV f1 =0.4992 +/- 0.0169
Holdout acc=0.5453 | macro-F1=0.5422
Saved confusion matrix: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/figures/task1/lr_handcrafted_holdout_confusion.png

=== svm_rbf_handcrafted ===
CV acc=0.5093 +/- 0.0097
CV f1 =0.5144 +/- 0.0111
Holdout acc=0.5200 | macro-F1=0.5251
Saved confusion matrix: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/figures/task1/svm_rbf_handcrafted_holdout_confusion.png

=== rf_handcrafted ===
CV acc=0.5110 +/- 0.0192
CV f1 =0.5047 +/- 0.0186
Holdout acc=0.5200 | macro-F1=0.5148
Saved confusion matrix: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/figures/task1/rf_handcrafted_holdout_confusion.png

=== lr_resnet18 ===


/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in ma

CV acc=0.8467 +/- 0.0076
CV f1 =0.8477 +/- 0.0072
Holdout acc=0.8640 | macro-F1=0.8651
Saved confusion matrix: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/figures/task1/lr_resnet18_holdout_confusion.png


/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [4]:
results_df = pd.DataFrame(results).sort_values(
    by=["holdout_macro_f1", "cv_macro_f1_mean"],
    ascending=False,
).reset_index(drop=True)

results_df

,model,feature_set,cv_acc_mean,cv_acc_std,cv_macro_f1_mean,cv_macro_f1_std,holdout_acc,holdout_macro_f1,fit_time_s,confusion_png
0,lr_resnet18,resnet18_512d,0.846667,0.007601,0.847740,0.007222,0.864000,0.865104,0.105357,outputs/figures/task1/lr_resnet18_holdout_confusion.png
1,lr_handcrafted,handcrafted_219d,0.501000,0.016418,0.499171,0.016880,0.545333,0.542151,0.158147,outputs/figures/task1/lr_handcrafted_holdout_confusion.png
2,svm_rbf_handcrafted,handcrafted_219d,0.509333,0.009695,0.514434,0.011091,0.520000,0.525142,1.495207,outputs/figures/task1/svm_rbf_handcrafted_holdout_confusion.png
3,rf_handcrafted,handcrafted_219d,0.511000,0.019166,0.504706,0.018555,0.520000,0.514783,1.809339,outputs/figures/task1/rf_handcrafted_holdout_confusion.png


In [5]:
submission_rows = []

for model_name, spec in model_specs.items():
    print(f"Training full-data model for submission: {model_name}")
    est = clone(spec["estimator"])

    t0 = time.perf_counter()
    est.fit(spec["X_train_full"], y)
    full_fit_s = time.perf_counter() - t0

    y_test_pred = est.predict(spec["X_test"])
    out_csv = OUT_PRED / f"task1_{model_name}_class_id.csv"
    make_submission(
        test_ids=test_ids,
        y_pred=y_test_pred,
        class_mapping=class_mapping,
        out_path=out_csv,
        label_column="class_id",
    )

    submission_rows.append(
        {
            "model": model_name,
            "submission_csv": str(out_csv.relative_to(REPO_ROOT)),
            "full_fit_time_s": full_fit_s,
        }
    )

submission_df = pd.DataFrame(submission_rows)
submission_df

Training full-data model for submission: lr_handcrafted


/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in ma

Training full-data model for submission: svm_rbf_handcrafted
Training full-data model for submission: rf_handcrafted
Training full-data model for submission: lr_resnet18


/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in ma

,model,submission_csv,full_fit_time_s
0,lr_handcrafted,outputs/predictions/task1_lr_handcrafted_class_id.csv,0.214923
1,svm_rbf_handcrafted,outputs/predictions/task1_svm_rbf_handcrafted_class_id.csv,2.060606
2,rf_handcrafted,outputs/predictions/task1_rf_handcrafted_class_id.csv,2.500668
3,lr_resnet18,outputs/predictions/task1_lr_resnet18_class_id.csv,0.162554


In [6]:
best_model = results_df.iloc[0]["model"]
best_src = OUT_PRED / f"task1_{best_model}_class_id.csv"
canonical_path = OUT_PRED / "task1_submission_class_id.csv"

canonical_df = pd.read_csv(best_src)
canonical_df.to_csv(canonical_path, index=False)

print(f"Best holdout model: {best_model}")
print(f"Canonical submission: {canonical_path}")
print("Submission preview:")
canonical_df.head()

Best holdout model: lr_resnet18
Canonical submission: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/predictions/task1_submission_class_id.csv
Submission preview:


,image_id,class_id
0,test_00000,9
1,test_00001,9
2,test_00002,9
3,test_00003,7
4,test_00004,7
